In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import os

os.makedirs("../output", exist_ok=True)

panel = pd.read_csv("../data/stacked_event_panel.csv")
panel = panel[(panel["k"] >= -6) & (panel["k"] <= 12)].copy()
panel["post"] = (panel["k"] >= 0).astype(int)

output_lines = []
def log(s=""):
    print(s)
    output_lines.append(s)

log("=" * 100)
log("TIMED-TO-MAJOR-EVENT vs. UNTIMED SPLIT")
log("=" * 100)
log(f"Timed events (Gronkowski/FanDuel, Kevin Hart/DraftKings, Jamie Foxx/BetMGM): "
    f"{(panel['timed_to_major_event']).sum()} obs, {panel[panel['timed_to_major_event']]['state_event'].nunique()} state-event units")
log(f"Untimed events (Barkley/FanDuel, Shaq/WynnBET, Mannings/Caesars): "
    f"{(~panel['timed_to_major_event']).sum()} obs, {panel[~panel['timed_to_major_event']]['state_event'].nunique()} state-event units\n")

for flag, label in [(True, "Timed to major sporting event"), (False, "Not timed to major sporting event")]:
    sub = panel[panel["timed_to_major_event"] == flag].copy()
    n_states = sub["state"].nunique()
    m_cluster = smf.ols("log_handle ~ post + C(state_event)", data=sub).fit(
        cov_type="cluster", cov_kwds={"groups": sub["state"]}
    )
    m_hc1 = smf.ols("log_handle ~ post + C(state_event)", data=sub).fit(cov_type="HC1")
    log(f"{label} (N={len(sub)}, {n_states} states, {sub['state_event'].nunique()} state-event units)")
    log(f"  post coefficient : {m_cluster.params['post']:+.3f}")
    log(f"  SE (cluster/state, {n_states} clusters): {m_cluster.bse['post']:.3f}   p={m_cluster.pvalues['post']:.3f}")
    log(f"  SE (HC1, non-clustered)                : {m_hc1.bse['post']:.3f}   p={m_hc1.pvalues['post']:.3f}")
    log(f"  implied handle change: {(np.exp(m_cluster.params['post']) - 1) * 100:.1f}%")
    log(f"  R-squared: {m_cluster.rsquared:.3f}   df_resid: {int(m_cluster.df_resid)}\n")

with open("../output/timed_vs_untimed_summary.txt", "w") as f:
    f.write("\n".join(output_lines))
print("Saved: ../output/timed_vs_untimed_summary.txt")

TIMED-TO-MAJOR-EVENT vs. UNTIMED SPLIT
Timed events (Gronkowski/FanDuel, Kevin Hart/DraftKings, Jamie Foxx/BetMGM): 152 obs, 8 state-event units
Untimed events (Barkley/FanDuel, Shaq/WynnBET, Mannings/Caesars): 72 obs, 4 state-event units

Timed to major sporting event (N=152, 3 states, 8 state-event units)
  post coefficient : +0.223
  SE (cluster/state, 3 clusters): 0.006   p=0.000
  SE (HC1, non-clustered)                : 0.047   p=0.000
  implied handle change: 25.0%
  R-squared: 0.825   df_resid: 143

Not timed to major sporting event (N=72, 2 states, 4 state-event units)
  post coefficient : +0.557
  SE (cluster/state, 2 clusters): 0.080   p=0.000
  SE (HC1, non-clustered)                : 0.118   p=0.000
  implied handle change: 74.6%
  R-squared: 0.512   df_resid: 67

Saved: ../output/timed_vs_untimed_summary.txt
